In [7]:
import pandas as pd
from pathlib import Path
from datetime import datetime

In [8]:
PROJECT_DIR = Path("/home/sc4r3cr0w/Schreibtisch/Data Analytics/WBS_Final_Project")
RAW = PROJECT_DIR / "RAW"
CLEAN_DIR = PROJECT_DIR / "CLEAN"
CLEAN_DIR.mkdir(exist_ok=True)

INCOME_DIR = RAW / "income"

print(INCOME_DIR.exists(), len(list(INCOME_DIR.glob('*'))))

True 2


In [9]:
df = pd.read_excel(
    INCOME_DIR / "entgelt-dwolk-0-202512-xlsx.xlsx",
    sheet_name="9.2",
    header=None,
)
print(df.shape)
df.iloc[7:14, :7]

(431, 11)


,0,1,2,3,4,5,6
7,Reg. Schlüs-\nsel,Region\n(Wohnort),"Median in €, jeweils zum Stichtag 31.12.",NaN,NaN,NaN,NaN
8,NaN,NaN,2021-12-31 00:00:00,2022-12-31 00:00:00,2023-12-31 00:00:00,2024-12-31 00:00:00,2025-12-31 00:00:00
9,NaN,NaN,1,2,3,4,5
10,D,Deutschland,3523.586991,3655.511486,3805.576112,4024.422191,4227.39957
11,W,Westdeutschland,3628.289975,3755.125284,3900.766011,4120.493101,4317.778772
12,O,Ostdeutschland,3059.128789,3214.710919,3388.810833,3600.284246,3824.908376
13,01,Schleswig-Holstein,3398.437087,3537.032181,3681.941156,3896.202317,4108.330338


In [11]:
date_cols = {}

for col in df.columns:
    for value in df[col]:
        if isinstance(value, datetime):
            date_cols[col] = value.year
            break

date_cols

{2: 2021, 3: 2022, 4: 2023, 5: 2024, 6: 2025}

In [12]:
is_district = df[0].astype(str).str.match(r"^\d{5}$")
districts = df[is_district]
print("Districts:", len(districts))

frames = []
for col, reference_year in date_cols.items():
    frame = pd.DataFrame({
        "year": reference_year + 1,
        "district_key": districts[0],
        "district_name": districts[1],
        "income_median_residence": districts[col],
    })
    frames.append(frame)

income = pd.concat(frames, ignore_index=True)
print("Rows:", len(income))
income.head()

Districts: 401
Rows: 2005


,year,district_key,district_name,income_median_residence
0,2022,01001,"Flensburg, Stadt",2927.171733
1,2022,01002,"Kiel, Landeshauptstadt",3459.520311
2,2022,01003,"Lübeck, Hansestadt",3332.2
3,2022,01004,"Neumünster, Stadt",3079.276042
4,2022,01051,Dithmarschen,3228.380658


In [13]:
income["income_median_residence"] = pd.to_numeric(income["income_median_residence"])

print(income.dtypes)
print(income.isna().sum())

income.groupby("year")["income_median_residence"].agg(["count", "min", "median", "max"]).round(0)

year                         int64
district_key                   str
district_name                  str
income_median_residence    float64
dtype: object
year                       0
district_key               0
district_name              0
income_median_residence    0
dtype: int64


,count,min,median,max
year,,,,
2022,401,2630.0,3461.0,4560.0
2023,401,2773.0,3582.0,4675.0
2024,401,2927.0,3724.0,4865.0
2025,401,3098.0,3935.0,5078.0
2026,401,3278.0,4132.0,5325.0


In [14]:
income.to_csv(CLEAN_DIR / "income_residence.csv", index=False)
print("Saved:", CLEAN_DIR / "income_residence.csv", "| rows:", len(income))

Saved: /home/sc4r3cr0w/Schreibtisch/Data Analytics/WBS_Final_Project/CLEAN/income_residence.csv | rows: 2005
